# Mask R-CNN — Clean Publication Pipeline (Kaggle)

**Correct pipeline:** YOLO crops → SAM masks → Manual verification → Paired augmentation → Mask R-CNN

## Kaggle Setup

1. **Upload dataset** as Kaggle Dataset: upload `yolo-crop-sam/` folder
   - Must contain: `images/`, `masks/`, `annotations.json`
   - Name it e.g. `mp-yolo-crop-sam`

2. **Add dataset** to notebook via sidebar → Add Data

3. **Enable GPU** (Settings → Accelerator → GPU T4 x2)

4. **Enable Internet** (Settings → Internet → On)

## Where Models Are Saved

```
/kaggle/working/
├── experiments/
│   ├── maskrcnn_crops_best.pth     ← DOWNLOAD THIS
│   └── maskrcnn_crops_latest.pth
└── mp_data/yolo-crop-sam/           ← local copy of data
```

After training: **Save Version** → download from Output tab.

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchvision albumentations pycocotools tqdm matplotlib

In [ ]:
import json, random, time, shutil
from pathlib import Path
from collections import defaultdict

import cv2, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision import transforms as T
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Embedded Functions

In [ ]:
NUM_CLASSES = 4
CLASS_NAMES = ['background', 'fiber', 'film', 'fragment']
YOLO_TO_MASKRCNN = {0: 1, 1: 2, 2: 3}


class CropDataset(Dataset):
    CLASS_NAME_TO_YOLO_ID = {'fiber': 0, 'film': 1, 'fragment': 2}

    def __init__(self, crops_dir, transforms=None, split_filter=None, sample_keys=None):
        self.crops_dir = Path(crops_dir)
        self.transforms = transforms
        ann_file = self.crops_dir / 'annotations.json'
        has_flat = (self.crops_dir / 'images').is_dir()
        has_cls = any((self.crops_dir / c).is_dir() for c in self.CLASS_NAME_TO_YOLO_ID)

        if ann_file.exists():
            with open(ann_file) as f: self.annotations = json.load(f)
            self.images_dir = (self.crops_dir / 'images') if has_flat else None
        elif has_cls:
            self.annotations = {}
            self.images_dir = None
            for cn, ci in self.CLASS_NAME_TO_YOLO_ID.items():
                cd = self.crops_dir / cn
                if not cd.is_dir(): continue
                for f in sorted(cd.glob('*.png')):
                    img = cv2.imread(str(f))
                    if img is None: continue
                    h, w = img.shape[:2]
                    self.annotations[f.name] = {'source_image': '', 'class_id': ci,
                        'class_name': cn, 'yolo_confidence': 1.0, 'rel_box': [0,0,w,h], 'crop_size': [w,h]}
        else:
            raise FileNotFoundError(f'No annotations or class dirs in {crops_dir}')

        if sample_keys is not None:
            self.samples = [k for k in sample_keys if k in self.annotations]
        elif split_filter:
            self.samples = [k for k,v in self.annotations.items() if v.get('split','train')==split_filter]
        else:
            self.samples = list(self.annotations.keys())
        print(f'CropDataset: {len(self.samples)} samples' + (f' (split={split_filter})' if split_filter else ''))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        name = self.samples[idx]
        ann = self.annotations[name]
        img_path = (self.images_dir / name) if self.images_dir else (self.crops_dir / ann.get('class_name','') / name)
        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        mask = None
        masks_dir = self.crops_dir / 'masks'
        mf = ann.get('mask_file')
        if mf and (masks_dir/mf).exists():
            raw = cv2.imread(str(masks_dir/mf), cv2.IMREAD_GRAYSCALE)
            if raw is not None: mask = (raw>127).astype(np.uint8)
        if mask is None:
            dm = masks_dir / name.replace('.png','_mask.png')
            if dm.exists():
                raw = cv2.imread(str(dm), cv2.IMREAD_GRAYSCALE)
                if raw is not None: mask = (raw>127).astype(np.uint8)
        if mask is None:
            mask = np.zeros((h,w), np.uint8)
            rb = ann.get('rel_box')
            cx,cy = ((rb[0]+rb[2])//2,(rb[1]+rb[3])//2) if rb else (w//2,h//2)
            ax,ay = ((rb[2]-rb[0])//2,(rb[3]-rb[1])//2) if rb else (int(w*0.4),int(h*0.4))
            if ax>0 and ay>0: cv2.ellipse(mask,(cx,cy),(ax,ay),0,0,360,1,-1)
        if mask.shape[:2]!=(h,w): mask = cv2.resize(mask,(w,h),interpolation=cv2.INTER_NEAREST)

        ys,xs = np.where(mask>0)
        box = [xs.min(),ys.min(),xs.max(),ys.max()] if len(xs)>0 else [min(h,w)//10]*2+[w-min(h,w)//10,h-min(h,w)//10]
        class_id = YOLO_TO_MASKRCNN[ann['class_id']]
        boxes = np.array([box], dtype=np.float32)
        labels = np.array([class_id], dtype=np.int64)
        masks = np.array([mask], dtype=np.uint8)

        if self.transforms:
            t = self.transforms(image=image, bboxes=boxes.tolist(), masks=list(masks), class_labels=labels.tolist())
            image = t['image']
            if len(t['bboxes'])>0:
                boxes = np.array(t['bboxes'], np.float32)
                labels = np.array(t['class_labels'], np.int64)
                masks = np.array(t['masks'], np.uint8)
        else:
            image = torch.from_numpy(image.transpose(2,0,1)).float()/255.0

        return image, {'boxes': torch.as_tensor(boxes, dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64),
            'masks': torch.as_tensor(masks, dtype=torch.uint8),
            'image_id': torch.tensor([idx]),
            'area': torch.as_tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes], dtype=torch.float32),
            'iscrowd': torch.zeros(len(boxes), dtype=torch.int64)}


def get_transforms(train=True, img_size=128):
    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5),
            A.ElasticTransform(alpha=30, sigma=5, p=0.2),
            A.RandomBrightnessContrast(0.3, 0.3, p=0.5),
            A.HueSaturationValue(10, 20, 20, p=0.3),
            A.GaussNoise(var_limit=(10.,50.), p=0.3),
            A.GaussianBlur(blur_limit=(3,5), p=0.2),
            A.CoarseDropout(max_holes=4, max_height=img_size//8, max_width=img_size//8, p=0.3),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2(),
        ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]), ToTensorV2(),
    ], bbox_params=A.BboxParams('pascal_voc', label_fields=['class_labels'], min_visibility=0.3))


def collate_fn(batch): return tuple(zip(*batch))

def get_model(num_classes, pretrained=True):
    w = MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn(weights=w)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes)
    inf_m = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(inf_m, 256, num_classes)
    return model

def freeze_backbone(model, freeze=True):
    for p in model.backbone.parameters(): p.requires_grad = not freeze
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Backbone {'frozen' if freeze else 'unfrozen'} ({n:,} trainable)")

@torch.no_grad()
def evaluate(model, loader, device):
    model.train()
    total, comp, count = 0., defaultdict(float), 0
    for imgs, tgts in loader:
        imgs = [i.to(device) for i in imgs]
        tgts = [{k:v.to(device) for k,v in t.items()} for t in tgts]
        if not all(len(t['boxes'])>0 for t in tgts): continue
        ld = model(imgs, tgts)
        total += sum(v.item() for v in ld.values())
        for k,v in ld.items(): comp[k] += v.item()
        count += 1
    avg = total/max(count,1)
    return avg, {k:v/max(count,1) for k,v in comp.items()}

print('All functions defined')

## 3. Copy Dataset to Working Directory

In [ ]:
import os

# >>> CHANGE THIS <<<
KAGGLE_DATASET_NAME = "mp-yolo-crop-sam"

INPUT_PATH = Path(f'/kaggle/input/{KAGGLE_DATASET_NAME}')
LOCAL_ROOT = Path('/kaggle/working/mp_data')
SAM_DIR    = LOCAL_ROOT / 'yolo-crop-sam'
SAVE_DIR   = Path('/kaggle/working/experiments')

assert INPUT_PATH.exists(), f"Dataset not found: {INPUT_PATH}\nAvailable: {os.listdir('/kaggle/input/')}"

# Auto-detect: might be directly in root or in subfolder
if (INPUT_PATH / 'annotations.json').exists():
    SRC = INPUT_PATH
elif (INPUT_PATH / 'yolo-crop-sam' / 'annotations.json').exists():
    SRC = INPUT_PATH / 'yolo-crop-sam'
else:
    found = list(INPUT_PATH.rglob('annotations.json'))
    assert found, f"No annotations.json found under {INPUT_PATH}"
    SRC = found[0].parent

if not SAM_DIR.exists():
    print(f'Copying {SRC} → {SAM_DIR}...')
    shutil.copytree(str(SRC), str(SAM_DIR))
    print('Done.')

SAVE_DIR.mkdir(parents=True, exist_ok=True)

CROP_SIZE = 128
MASKRCNN_BATCH = 8
PHASE1_EPOCHS = 5; PHASE1_LR = 5e-4
PHASE2_EPOCHS = 20; PHASE2_LR = 1e-4
WEIGHT_DECAY = 0.005
EARLY_STOP_PATIENCE = 7
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device: {DEVICE}, Data: {SAM_DIR}, Save: {SAVE_DIR}')

## 4. Explore Dataset

In [ ]:
with open(SAM_DIR / 'annotations.json') as f:
    annotations = json.load(f)

n_imgs = len(list((SAM_DIR/'images').glob('*.png')))
n_masks = len(list((SAM_DIR/'masks').glob('*.png')))
print(f'Images: {n_imgs}, Masks: {n_masks}, Annotations: {len(annotations)}')

cls_counts = defaultdict(int)
for ann in annotations.values(): cls_counts[ann.get('class_name','?')] += 1
for c,n in sorted(cls_counts.items()): print(f'  {c:10s}: {n:5d} ({100*n/len(annotations):.1f}%)')

## 5. Train/Val Split & Build Datasets

In [ ]:
with open(SAM_DIR / 'annotations.json') as f:
    all_ann = json.load(f)

has_split = any(v.get('split') for v in all_ann.values())
if has_split:
    train_keys = [k for k,v in all_ann.items() if v.get('split')=='train']
    val_keys = [k for k,v in all_ann.items() if v.get('split')=='val']
else:
    src_to_keys = defaultdict(list)
    for k,v in all_ann.items(): src_to_keys[v.get('source_image',k)].append(k)
    sources = sorted(src_to_keys.keys())
    random.seed(42); random.shuffle(sources)
    n_train = max(1, int(len(sources)*0.8))
    train_keys = [k for s in sources[:n_train] for k in src_to_keys[s]]
    val_keys = [k for s in sources[n_train:] for k in src_to_keys[s]]

print(f'Train: {len(train_keys)}, Val: {len(val_keys)}')

train_ds = CropDataset(str(SAM_DIR), get_transforms(True, CROP_SIZE), sample_keys=train_keys)
val_ds = CropDataset(str(SAM_DIR), get_transforms(False, CROP_SIZE), sample_keys=val_keys)

## 6. Model & DataLoaders

In [ ]:
model = get_model(NUM_CLASSES, pretrained=True).to(DEVICE)
print(f'Mask R-CNN: {sum(p.numel() for p in model.parameters()):,} params')

train_loader = DataLoader(train_ds, MASKRCNN_BATCH, shuffle=True, num_workers=2, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, MASKRCNN_BATCH, shuffle=False, num_workers=2, collate_fn=collate_fn)
print(f'Train: {len(train_loader)} batches, Val: {len(val_loader)} batches')

## 7. Train (2-Phase + Early Stopping)

In [ ]:
best_val_loss = float('inf')
patience_counter = 0
global_epoch = 0
history = {'epoch':[], 'train_loss':[], 'val_loss':[], 'lr':[], 'phase':[]}

# Auto-backup every 5 epochs
BACKUP_EVERY = 5
BACKUP_ROOT  = Path('/kaggle/working/backups')

def run_phase(name, n_epochs, lr, freeze_bb):
    global best_val_loss, patience_counter, global_epoch
    freeze_backbone(model, freeze_bb)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    print(f"\n{'='*60}\n{name}: {n_epochs} ep, LR={lr}\n{'='*60}")
    for ep in range(1, n_epochs+1):
        global_epoch += 1
        model.train(); epoch_loss = 0.; n_batch = 0
        for images, targets in tqdm(train_loader, desc=f'[{name}] Ep {ep}/{n_epochs}', leave=False):
            images = [i.to(DEVICE) for i in images]
            targets = [{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
            if not all(len(t['boxes'])>0 for t in targets): continue
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())
            optimizer.zero_grad(); losses.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0); optimizer.step()
            epoch_loss += losses.item(); n_batch += 1

        avg_train = epoch_loss / max(n_batch,1)
        val_loss, _ = evaluate(model, val_loader, DEVICE)
        scheduler.step(val_loss)
        history['epoch'].append(global_epoch)
        history['train_loss'].append(avg_train)
        history['val_loss'].append(val_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        history['phase'].append(name)

        ckpt = {'epoch': global_epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(), 'val_loss': val_loss,
                'history': history, 'phase1_epochs': PHASE1_EPOCHS}
        torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_latest.pth'))
        tag = ''
        if val_loss < best_val_loss:
            best_val_loss = val_loss; patience_counter = 0
            torch.save(ckpt, str(SAVE_DIR / 'maskrcnn_crops_best.pth'))
            tag = ' * best'
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f'  Early stopping at ep {ep}'); return True

        # Auto-backup
        if global_epoch % BACKUP_EVERY == 0:
            backup_dir = BACKUP_ROOT / f'epoch_{global_epoch:04d}'
            backup_dir.mkdir(parents=True, exist_ok=True)
            for bname in ('maskrcnn_crops_best.pth', 'maskrcnn_crops_latest.pth'):
                src = SAVE_DIR / bname
                if src.exists():
                    shutil.copy2(str(src), str(backup_dir / bname))
            print(f'  [AUTO-BACKUP] epoch {global_epoch} → {backup_dir}')

        if ep % 2 == 0 or ep == 1:
            print(f'  Ep {ep}/{n_epochs} train={avg_train:.4f} val={val_loss:.4f}{tag}')
    return False

print(f'Auto-backup: every {BACKUP_EVERY} epochs → {BACKUP_ROOT}')
stopped = run_phase('Phase 1 (heads)', PHASE1_EPOCHS, PHASE1_LR, True)
if not stopped:
    run_phase('Phase 2 (full)', PHASE2_EPOCHS, PHASE2_LR, False)

print(f"\nDone — {global_epoch} epochs, best val={best_val_loss:.4f}")
print(f"Best model: {SAVE_DIR / 'maskrcnn_crops_best.pth'}")


## 8. Training Curves

In [ ]:
# ==============================================================================
# 8. TRAINING CURVES (self-contained)
# ==============================================================================

import torch, matplotlib.pyplot as plt
from pathlib import Path

SAVE_DIR    = Path('/kaggle/working/experiments')
BACKUP_ROOT = Path('/kaggle/working/backups')

def find_best_checkpoint(save_dir, backup_root):
    primary = save_dir / 'maskrcnn_crops_best.pth'
    if primary.exists():
        return str(primary)
    backups = sorted(backup_root.glob('epoch_*/maskrcnn_crops_best.pth')) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path('/kaggle/working').rglob('maskrcnn_crops_best.pth'))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError('maskrcnn_crops_best.pth not found.')

ckpt_path = find_best_checkpoint(SAVE_DIR, BACKUP_ROOT)
ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
history = ckpt['history']
print(f'Loaded history from {ckpt_path} (epoch {ckpt["epoch"]})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history['epoch'], history['train_loss'], 'b-', label='Train')
ax1.plot(history['epoch'], history['val_loss'], 'r-', label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(history['epoch'], history['lr'], 'g-')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('LR'); ax2.set_title('Learning Rate'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 9. Visualise Predictions

In [ ]:
# ==============================================================================
# 9. VISUALISE PREDICTIONS (self-contained)
# ==============================================================================

import json, random, torch, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from torchvision import transforms as T
from torchvision.models.detection import maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

NUM_CLASSES = 4
CLASS_NAMES = ['background', 'fiber', 'film', 'fragment']
CROP_SIZE   = 128
SAVE_DIR    = Path('/kaggle/working/experiments')
SAM_DIR     = Path('/kaggle/working/mp_data/yolo-crop-sam')
BACKUP_ROOT = Path('/kaggle/working/backups')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_best_checkpoint(save_dir, backup_root):
    primary = save_dir / 'maskrcnn_crops_best.pth'
    if primary.exists():
        return str(primary)
    backups = sorted(backup_root.glob('epoch_*/maskrcnn_crops_best.pth')) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path('/kaggle/working').rglob('maskrcnn_crops_best.pth'))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError('maskrcnn_crops_best.pth not found.')

def get_model(num_classes):
    m = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, num_classes)
    inf_m = m.roi_heads.mask_predictor.conv5_mask.in_channels
    m.roi_heads.mask_predictor = MaskRCNNPredictor(inf_m, 256, num_classes)
    return m

ckpt_path = find_best_checkpoint(SAVE_DIR, BACKUP_ROOT)
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model = get_model(NUM_CLASSES).to(DEVICE)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
print(f'Loaded best model (epoch {ckpt["epoch"]})')

inference_tf = T.Compose([T.ToPILImage(), T.Resize((CROP_SIZE,CROP_SIZE)),
                          T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

with open(SAM_DIR / 'annotations.json') as f: test_anns = json.load(f)
samples = random.sample(list(test_anns.keys()), min(6, len(test_anns)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, name in zip(axes.flatten(), samples):
    ann = test_anns[name]
    img = cv2.cvtColor(cv2.imread(str(SAM_DIR/'images'/name)), cv2.COLOR_BGR2RGB)
    inp = inference_tf(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): pred = model(inp)[0]
    img_r = cv2.resize(img, (CROP_SIZE,CROP_SIZE))
    keep = pred['scores'] > 0.3
    if keep.sum() > 0:
        mask = pred['masks'][keep][0,0].cpu().numpy() > 0.5
        score = pred['scores'][keep][0].item()
        label = CLASS_NAMES[pred['labels'][keep][0].item()]
        overlay = img_r.copy().astype(float)/255
        overlay[mask] = overlay[mask]*0.5 + np.array([0,1,0])*0.5
        ax.imshow(overlay); ax.set_title(f'{label} ({score:.2f})')
    else:
        ax.imshow(img_r); ax.set_title('No detection')
    ax.axis('off')
plt.tight_layout(); plt.show()


## 10. Save & Download

Models are already saved in `/kaggle/working/experiments/`.
Click **Save Version** then download from the **Output** tab.

In [ ]:
# ==============================================================================
# 10. SAVE & DOWNLOAD (self-contained)
# ==============================================================================

import shutil
from pathlib import Path

SAVE_DIR    = Path('/kaggle/working/experiments')
BACKUP_ROOT = Path('/kaggle/working/backups')

def find_best_checkpoint(save_dir, backup_root):
    primary = save_dir / 'maskrcnn_crops_best.pth'
    if primary.exists():
        return str(primary)
    backups = sorted(backup_root.glob('epoch_*/maskrcnn_crops_best.pth')) if backup_root.exists() else []
    if backups:
        return str(backups[-1])
    candidates = sorted(Path('/kaggle/working').rglob('maskrcnn_crops_best.pth'))
    if candidates:
        return str(candidates[-1])
    raise FileNotFoundError('maskrcnn_crops_best.pth not found.')

ckpt_path = find_best_checkpoint(SAVE_DIR, BACKUP_ROOT)
shutil.copy2(ckpt_path, '/kaggle/working/maskrcnn_crops_best.pth')

for f in SAVE_DIR.glob('*.pth'):
    print(f'{f.name}: {f.stat().st_size/1e6:.1f} MB')

print(f'\nCopied from: {ckpt_path}')
print('Save Version → download from Output tab')
